# 2026H1 任务总览与可行性评估

本 notebook 对任务书目标进行可执行拆解，并给出当前仓库可直接完成与需近似处理的边界。

## 可行性结论

1. 可直接做：Lindblad/T1/T2 建模、1/f 与 OU 等效噪声、单/多比特门噪声影响分析、参数敏感性扫描、五比特误差传播演示。
2. 需近似做：AWG 失真与完整测控链路先用等效参数（control_scale、gate_duration、噪声注入）建模。
3. 建议交付：按 01-06 notebook 顺序执行，最后汇总图表形成季度报告。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'examples' / 'backend.yaml').exists():
            return p
    raise FileNotFoundError('???????????? pyproject.toml ? examples/backend.yaml?')

ROOT = find_project_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from qsim.ui.notebook import run_workflow

BACKEND_PATH = ROOT / 'examples' / 'backend.yaml'
OUT_ROOT = ROOT / 'examples' / 'noise_simulation_tests' / 'runs' / 'roadmap_2026H1'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('BACKEND_PATH =', BACKEND_PATH)
print('OUT_ROOT =', OUT_ROOT)


def run_case(tag: str, qasm_text: str, hardware: dict | None = None, noise: dict | None = None, engine: str = 'qutip'):
    out_dir = OUT_ROOT / tag
    return run_workflow(
        qasm_text=qasm_text,
        backend_path=str(BACKEND_PATH),
        out_dir=str(out_dir),
        hardware=hardware or {},
        noise=noise or {},
        engine=engine,
        persist_artifacts=True,
        export_dxf=False,
    )


def get_metric(result: dict, key: str, default: float = np.nan) -> float:
    obs = result.get('analysis', {}).get('observables', {}).get('values', {})
    return float(obs.get(key, default))

## 月度 notebook 对应关系

- 01：噪声源分类与建模框架（1 月）
- 02：单比特 Lindblad + T1/T2 + 1/f（2 月）
- 03：控制与读取噪声（AWG/链路响应）（3 月）
- 04：两比特耦合与串扰（4 月）
- 05：两比特误差敏感性与可视化（5 月）
- 06：五比特噪声传播与闭环仿真（6 月）
